# Outlier Detection

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadtalhaishtiaq/ai-orchestrator/blob/main/02-exploratory-data-analysis/05_outlier_detection.ipynb)

## Learning Objectives
- Understand what outliers are and why they matter
- Master statistical methods for outlier detection
- Learn visualization techniques for spotting outliers
- Know when to remove vs keep outliers

---

## 1. What are Outliers?

**Outlier** = A data point that differs significantly from other observations

**Types of Outliers:**
1. **Point outliers**: Single extreme value (most common)
2. **Contextual outliers**: Abnormal in specific context (e.g., 30°C in winter)
3. **Collective outliers**: Group of values that's anomalous together

**Why Outliers Matter:**
- ✅ **Can be valuable**: Fraud detection, rare events, discoveries
- ❌ **Can be problematic**: Skew statistics, distort models, indicate errors

**Causes:**
- 📊 Natural variation (real extreme values)
- ⌨️ Data entry errors
- 🔧 Measurement errors
- 🧪 Experimental errors
- 📉 Sampling errors

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
%matplotlib inline

## 2. Dataset: Credit Card Transactions

In [ ]:
# Create synthetic credit card transaction dataset
np.random.seed(42)
n = 500

# Normal transactions
normal_amounts = np.random.gamma(50, 2, int(n * 0.95))  # Most transactions

# Add some outliers (fraudulent or high-value transactions)
outlier_amounts = np.random.uniform(500, 5000, int(n * 0.05))

# Combine
amounts = np.concatenate([normal_amounts, outlier_amounts])
np.random.shuffle(amounts)

# Create other features
df = pd.DataFrame({
    'transaction_id': range(1, n+1),
    'amount': amounts.round(2),
    'merchant_category': np.random.choice(['Retail', 'Food', 'Transport', 'Entertainment', 'Online'], n),
    'hour_of_day': np.random.randint(0, 24, n),
    'day_of_week': np.random.randint(0, 7, n),
    'distance_from_home_km': np.abs(np.random.normal(5, 10, n)).round(1),
    'transaction_count_last_24h': np.random.poisson(3, n)
})

# Add some multivariate outliers (unusual combinations)
outlier_idx = np.random.choice(n, 10, replace=False)
df.loc[outlier_idx, 'distance_from_home_km'] = np.random.uniform(100, 500, 10)
df.loc[outlier_idx, 'hour_of_day'] = np.random.choice([2, 3, 4], 10)  # Unusual hours

print("✅ Transaction dataset created!")
print(f"Shape: {df.shape}")
df.head(10)

## 3. Univariate Outlier Detection

### 3.1 Visualization Methods

In [ ]:
# Multiple visualizations for outlier detection
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Histogram
axes[0, 0].hist(df['amount'], bins=50, edgecolor='black', alpha=0.7, color='skyblue')
axes[0, 0].set_title('Histogram: Transaction Amount', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Amount ($)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].axvline(df['amount'].mean(), color='red', linestyle='--', linewidth=2, label='Mean')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# 2. Box Plot
box = axes[0, 1].boxplot(df['amount'], vert=True, patch_artist=True,
                          boxprops=dict(facecolor='lightgreen', alpha=0.7),
                          medianprops=dict(color='red', linewidth=2),
                          flierprops=dict(marker='o', markerfacecolor='red', markersize=8, alpha=0.5))
axes[0, 1].set_title('Box Plot: Transaction Amount', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('Amount ($)')
axes[0, 1].grid(alpha=0.3)

# 3. Scatter Plot (index vs value)
axes[1, 0].scatter(range(len(df)), df['amount'], alpha=0.5, s=30)
axes[1, 0].axhline(df['amount'].mean(), color='green', linestyle='--', linewidth=2, label='Mean')
axes[1, 0].axhline(df['amount'].mean() + 3*df['amount'].std(), color='red', linestyle='--', linewidth=1, label='+3σ')
axes[1, 0].axhline(df['amount'].mean() - 3*df['amount'].std(), color='red', linestyle='--', linewidth=1, label='-3σ')
axes[1, 0].set_title('Scatter: Transaction Amount by Index', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Transaction Index')
axes[1, 0].set_ylabel('Amount ($)')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# 4. Q-Q Plot (normality check)
stats.probplot(df['amount'], dist="norm", plot=axes[1, 1])
axes[1, 1].set_title('Q-Q Plot: Transaction Amount', fontsize=12, fontweight='bold')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("💡 Red points in box plot = Outliers detected by IQR method")

### 3.2 IQR (Interquartile Range) Method

In [ ]:
# IQR Method: Most common and robust
Q1 = df['amount'].quantile(0.25)
Q3 = df['amount'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers_iqr = df[(df['amount'] < lower_bound) | (df['amount'] > upper_bound)]

print("📊 IQR Method Results:")
print(f"Q1 (25th percentile): ${Q1:.2f}")
print(f"Q3 (75th percentile): ${Q3:.2f}")
print(f"IQR: ${IQR:.2f}")
print(f"\nLower Bound: ${lower_bound:.2f}")
print(f"Upper Bound: ${upper_bound:.2f}")
print(f"\n🚨 Outliers Detected: {len(outliers_iqr)} ({len(outliers_iqr) / len(df) * 100:.1f}%)")

if len(outliers_iqr) > 0:
    print(f"\nOutlier Statistics:")
    print(f"Min outlier: ${outliers_iqr['amount'].min():.2f}")
    print(f"Max outlier: ${outliers_iqr['amount'].max():.2f}")
    print(f"Mean outlier: ${outliers_iqr['amount'].mean():.2f}")

### 3.3 Z-Score Method

In [ ]:
# Z-Score Method: Measures how many standard deviations away from mean
# Assumes normal distribution
z_scores = np.abs(stats.zscore(df['amount']))
threshold = 3  # Common threshold: |Z| > 3

outliers_z = df[z_scores > threshold]

print("📊 Z-Score Method Results:")
print(f"Mean: ${df['amount'].mean():.2f}")
print(f"Std Dev: ${df['amount'].std():.2f}")
print(f"Threshold: |Z| > {threshold}")
print(f"\n🚨 Outliers Detected: {len(outliers_z)} ({len(outliers_z) / len(df) * 100:.1f}%)")

# Visualize Z-scores
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(range(len(df)), z_scores, alpha=0.5, s=30)
plt.axhline(threshold, color='red', linestyle='--', linewidth=2, label=f'Threshold: {threshold}')
plt.xlabel('Transaction Index')
plt.ylabel('|Z-Score|')
plt.title('Z-Scores for Transaction Amounts', fontsize=12, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.hist(z_scores, bins=50, edgecolor='black', alpha=0.7, color='coral')
plt.axvline(threshold, color='red', linestyle='--', linewidth=2, label=f'Threshold: {threshold}')
plt.xlabel('|Z-Score|')
plt.ylabel('Frequency')
plt.title('Distribution of Z-Scores', fontsize=12, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 3.4 Modified Z-Score (Robust to Outliers)

In [ ]:
# Modified Z-Score: Uses median instead of mean (more robust)
median = df['amount'].median()
mad = np.median(np.abs(df['amount'] - median))  # Median Absolute Deviation
modified_z_scores = 0.6745 * (df['amount'] - median) / mad

outliers_mod_z = df[np.abs(modified_z_scores) > 3.5]

print("📊 Modified Z-Score Method Results:")
print(f"Median: ${median:.2f}")
print(f"MAD (Median Absolute Deviation): ${mad:.2f}")
print(f"\n🚨 Outliers Detected: {len(outliers_mod_z)} ({len(outliers_mod_z) / len(df) * 100:.1f}%)")

# Compare methods
print("\n📊 Method Comparison:")
comparison = pd.DataFrame({
    'Method': ['IQR', 'Z-Score', 'Modified Z-Score'],
    'Outliers': [len(outliers_iqr), len(outliers_z), len(outliers_mod_z)],
    'Percentage': [
        f"{len(outliers_iqr) / len(df) * 100:.1f}%",
        f"{len(outliers_z) / len(df) * 100:.1f}%",
        f"{len(outliers_mod_z) / len(df) * 100:.1f}%"
    ]
})
print(comparison.to_string(index=False))

## 4. Multivariate Outlier Detection

### 4.1 Mahalanobis Distance

In [ ]:
# Mahalanobis Distance: Considers correlations between variables
from scipy.spatial.distance import mahalanobis

# Select numerical features
features = ['amount', 'hour_of_day', 'distance_from_home_km', 'transaction_count_last_24h']
X = df[features].values

# Calculate mean and covariance
mean = np.mean(X, axis=0)
cov = np.cov(X.T)

# Calculate Mahalanobis distance for each point
try:
    inv_cov = np.linalg.inv(cov)
    mahal_distances = []
    for i in range(len(X)):
        dist = mahalanobis(X[i], mean, inv_cov)
        mahal_distances.append(dist)
    
    df['mahal_distance'] = mahal_distances
    
    # Threshold: Chi-square distribution with k degrees of freedom
    from scipy.stats import chi2
    threshold = chi2.ppf(0.95, df=len(features))  # 95% confidence
    
    outliers_mahal = df[df['mahal_distance'] > threshold]
    
    print("📊 Mahalanobis Distance Method:")
    print(f"Threshold (95% confidence): {threshold:.2f}")
    print(f"\n🚨 Outliers Detected: {len(outliers_mahal)} ({len(outliers_mahal) / len(df) * 100:.1f}%)")
    
    # Visualize
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.scatter(range(len(df)), df['mahal_distance'], alpha=0.5, s=30)
    plt.axhline(threshold, color='red', linestyle='--', linewidth=2, label=f'Threshold: {threshold:.2f}')
    plt.xlabel('Transaction Index')
    plt.ylabel('Mahalanobis Distance')
    plt.title('Mahalanobis Distance', fontsize=12, fontweight='bold')
    plt.legend()
    plt.grid(alpha=0.3)
    
    plt.subplot(1, 2, 2)
    plt.hist(df['mahal_distance'], bins=50, edgecolor='black', alpha=0.7, color='purple')
    plt.axvline(threshold, color='red', linestyle='--', linewidth=2, label=f'Threshold: {threshold:.2f}')
    plt.xlabel('Mahalanobis Distance')
    plt.ylabel('Frequency')
    plt.title('Distribution of Mahalanobis Distances', fontsize=12, fontweight='bold')
    plt.legend()
    plt.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
except np.linalg.LinAlgError:
    print("⚠️ Covariance matrix is singular - cannot compute Mahalanobis distance")

### 4.2 Isolation Forest

In [ ]:
# Isolation Forest: ML-based anomaly detection
# Idea: Outliers are easier to isolate (fewer splits needed)

# Prepare data
X = df[features].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train Isolation Forest
iso_forest = IsolationForest(contamination=0.05, random_state=42)  # Expect 5% outliers
outlier_labels = iso_forest.fit_predict(X_scaled)

# -1 = outlier, 1 = inlier
df['isolation_forest'] = outlier_labels
outliers_iso = df[df['isolation_forest'] == -1]

print("📊 Isolation Forest Method:")
print(f"Contamination: 5%")
print(f"\n🚨 Outliers Detected: {len(outliers_iso)} ({len(outliers_iso) / len(df) * 100:.1f}%)")

# Visualize (2D projection)
plt.figure(figsize=(10, 6))
inliers = df[df['isolation_forest'] == 1]
outliers = df[df['isolation_forest'] == -1]

plt.scatter(inliers['amount'], inliers['distance_from_home_km'],
            alpha=0.5, s=30, label='Inliers', color='blue')
plt.scatter(outliers['amount'], outliers['distance_from_home_km'],
            alpha=0.8, s=100, label='Outliers', color='red', marker='*', edgecolors='black')
plt.xlabel('Amount ($)', fontsize=12)
plt.ylabel('Distance from Home (km)', fontsize=12)
plt.title('Isolation Forest Outlier Detection', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 4.3 Local Outlier Factor (LOF)

In [ ]:
# LOF: Density-based outlier detection
# Idea: Outliers have lower density than their neighbors

lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05)
outlier_labels_lof = lof.fit_predict(X_scaled)

df['lof'] = outlier_labels_lof
outliers_lof = df[df['lof'] == -1]

print("📊 Local Outlier Factor Method:")
print(f"Neighbors: 20")
print(f"\n🚨 Outliers Detected: {len(outliers_lof)} ({len(outliers_lof) / len(df) * 100:.1f}%)")

# Visualize
plt.figure(figsize=(10, 6))
inliers_lof = df[df['lof'] == 1]
outliers_lof_vis = df[df['lof'] == -1]

plt.scatter(inliers_lof['hour_of_day'], inliers_lof['transaction_count_last_24h'],
            alpha=0.5, s=30, label='Inliers', color='green')
plt.scatter(outliers_lof_vis['hour_of_day'], outliers_lof_vis['transaction_count_last_24h'],
            alpha=0.8, s=100, label='Outliers', color='red', marker='*', edgecolors='black')
plt.xlabel('Hour of Day', fontsize=12)
plt.ylabel('Transaction Count (Last 24h)', fontsize=12)
plt.title('Local Outlier Factor Detection', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Outlier Treatment Decision Tree

```
Is the outlier a DATA ERROR?
    ├─ YES → Fix or remove it
    │
    └─ NO → Is it VALID but EXTREME?
        ├─ YES → Keep it, use robust methods
        │   Options:
        │   • Use robust models (tree-based, SVR)
        │   • Transform data (log, sqrt)
        │   • Use robust statistics (median, MAD)
        │   • Cap/floor values (winsorization)
        │
        └─ UNSURE → Analyze impact
            • Train model WITH outliers
            • Train model WITHOUT outliers
            • Compare performance
            • Choose better approach
```

## 6. Outlier Treatment Methods

### 6.1 Removal

In [ ]:
# Method 1: Remove outliers
df_no_outliers = df[z_scores <= 3].copy()

print("❌ Removal Method:")
print(f"Original size: {len(df)}")
print(f"After removal: {len(df_no_outliers)}")
print(f"Removed: {len(df) - len(df_no_outliers)} rows ({(len(df) - len(df_no_outliers)) / len(df) * 100:.1f}%)")

print(f"\nImpact on statistics:")
print(f"Original mean: ${df['amount'].mean():.2f}")
print(f"After removal: ${df_no_outliers['amount'].mean():.2f}")
print(f"Original std: ${df['amount'].std():.2f}")
print(f"After removal: ${df_no_outliers['amount'].std():.2f}")

### 6.2 Capping (Winsorization)

In [ ]:
# Method 2: Cap outliers at percentiles
lower_percentile = df['amount'].quantile(0.01)
upper_percentile = df['amount'].quantile(0.99)

df_capped = df.copy()
df_capped['amount_capped'] = df_capped['amount'].clip(lower=lower_percentile, upper=upper_percentile)

print("🔒 Capping Method (Winsorization):")
print(f"Lower cap (1st percentile): ${lower_percentile:.2f}")
print(f"Upper cap (99th percentile): ${upper_percentile:.2f}")
print(f"\nValues capped: {(df['amount'] != df_capped['amount_capped']).sum()}")

# Visualize before/after
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['amount'], bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[0].set_title('Before Capping', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Amount ($)')
axes[0].set_ylabel('Frequency')
axes[0].grid(alpha=0.3)

axes[1].hist(df_capped['amount_capped'], bins=50, edgecolor='black', alpha=0.7, color='green')
axes[1].set_title('After Capping', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Amount ($)')
axes[1].set_ylabel('Frequency')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 6.3 Transformation

In [ ]:
# Method 3: Transform data to reduce impact of outliers
df_transformed = df.copy()
df_transformed['amount_log'] = np.log1p(df['amount'])  # log(1 + x) to handle zeros
df_transformed['amount_sqrt'] = np.sqrt(df['amount'])

# Visualize transformations
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df['amount'], bins=50, edgecolor='black', alpha=0.7, color='blue')
axes[0].set_title('Original', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Amount ($)')
axes[0].set_ylabel('Frequency')
axes[0].grid(alpha=0.3)

axes[1].hist(df_transformed['amount_log'], bins=50, edgecolor='black', alpha=0.7, color='purple')
axes[1].set_title('Log Transformation', fontsize=12, fontweight='bold')
axes[1].set_xlabel('log(Amount)')
axes[1].set_ylabel('Frequency')
axes[1].grid(alpha=0.3)

axes[2].hist(df_transformed['amount_sqrt'], bins=50, edgecolor='black', alpha=0.7, color='teal')
axes[2].set_title('Square Root Transformation', fontsize=12, fontweight='bold')
axes[2].set_xlabel('sqrt(Amount)')
axes[2].set_ylabel('Frequency')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("📊 Transformation Impact on Skewness:")
print(f"Original: {df['amount'].skew():.3f}")
print(f"Log: {df_transformed['amount_log'].skew():.3f}")
print(f"Square Root: {df_transformed['amount_sqrt'].skew():.3f}")

## 7. Comprehensive Outlier Report

In [ ]:
def outlier_report(dataframe, column, show_plot=True):
    """
    Generate comprehensive outlier analysis report
    """
    print("="*60)
    print(f"📊 OUTLIER ANALYSIS: {column}")
    print("="*60)
    
    data = dataframe[column].dropna()
    
    # Basic stats
    print(f"\n📈 Basic Statistics:")
    print(f"Count: {len(data)}")
    print(f"Mean: {data.mean():.2f}")
    print(f"Median: {data.median():.2f}")
    print(f"Std Dev: {data.std():.2f}")
    print(f"Skewness: {data.skew():.3f}")
    
    # IQR Method
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers_iqr = data[(data < lower) | (data > upper)]
    
    print(f"\n🔍 IQR Method:")
    print(f"Outliers: {len(outliers_iqr)} ({len(outliers_iqr) / len(data) * 100:.1f}%)")
    
    # Z-Score Method
    z_scores = np.abs(stats.zscore(data))
    outliers_z = data[z_scores > 3]
    
    print(f"\n🔍 Z-Score Method (|Z| > 3):")
    print(f"Outliers: {len(outliers_z)} ({len(outliers_z) / len(data) * 100:.1f}%)")
    
    # Recommendations
    print(f"\n💡 Recommendations:")
    if len(outliers_iqr) > len(data) * 0.1:
        print("⚠️ High proportion of outliers - investigate data quality")
    if abs(data.skew()) > 1:
        print("📊 Highly skewed - consider log/sqrt transformation")
    if len(outliers_iqr) < 5:
        print("✅ Few outliers - safe to investigate individually")
    else:
        print("⚡ Many outliers - use robust methods or transformation")
    
    if show_plot:
        # Visualization
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Box plot
        axes[0].boxplot(data, vert=True)
        axes[0].set_title(f'Box Plot: {column}', fontsize=12, fontweight='bold')
        axes[0].set_ylabel(column)
        axes[0].grid(alpha=0.3)
        
        # Histogram
        axes[1].hist(data, bins=50, edgecolor='black', alpha=0.7)
        axes[1].axvline(data.mean(), color='red', linestyle='--', linewidth=2, label='Mean')
        axes[1].axvline(data.median(), color='green', linestyle='--', linewidth=2, label='Median')
        axes[1].set_title(f'Distribution: {column}', fontsize=12, fontweight='bold')
        axes[1].set_xlabel(column)
        axes[1].set_ylabel('Frequency')
        axes[1].legend()
        axes[1].grid(alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    
    print("\n" + "="*60)

# Run report
outlier_report(df, 'amount', show_plot=True)

## 8. Your Turn! 💪

**Exercise**: Analyze outliers in the `distance_from_home_km` column:
1. Use IQR and Z-score methods
2. Visualize with box plot and histogram
3. Try capping (winsorization)
4. Decide: Should you remove, cap, or keep these outliers? Why?

In [ ]:
# Your code here

---

## Key Takeaways 🎯

1. **Outliers aren't always errors**: Investigate before removing
2. **IQR method**: Robust, works for non-normal distributions
3. **Z-score method**: Assumes normality, sensitive to extreme values
4. **Multivariate methods**: Isolation Forest, LOF for complex cases
5. **Treatment options**: Remove, cap, transform, or use robust methods
6. **Always visualize**: Box plots and scatter plots are essential
7. **Domain knowledge**: Critical for deciding what to do with outliers

**Next**: Explore automated EDA tools to speed up analysis! 🚀